# Early warning — alertas de degradação no FD001

Uma previsão de RUL só se torna útil para manutenção quando pode ser transformada em uma decisão de alerta. Nesta etapa, será investigado quando o modelo emitiria um aviso e se esse aviso chegaria em tempo de planejamento.

Para evitar uma avaliação otimista, as previsões serão geradas fora da amostra. Cada motor permanecerá inteiro em uma única parte da validação, seguindo o mesmo princípio da validação por trajetória utilizado anteriormente (Pedregosa et al., 2011).

In [1]:
from pathlib import Path
import sys
import pandas as pd
import plotly.express as px
import plotly.io as pio
from multilang import LangMap

pio.renderers.default = 'vscode'
lang = LangMap(source='pt', target='pt')

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'data' / 'raw' / 'dataset').exists():
    PROJECT_ROOT = Path.cwd().resolve().parents[1]
if str(PROJECT_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from cmapss_eda import load_split
from cmapss_modeling import add_rul_label, candidate_rul_features, nonconstant_features
from cmapss_optuna import build_pipeline_from_params, optimize_model
from cmapss_early_warning import evaluate_warning_grid, generate_oof_rul_predictions

DATA_DIR = PROJECT_ROOT / 'data' / 'raw' / 'dataset'

## 1. Previsões fora da amostra

Será utilizado o Extra Trees v1, que foi o melhor candidato na validação anterior. Esse estimador combina várias árvores com aleatoriedade adicional na construção das regras, reduzindo a dependência de uma única árvore (Geurts et al., 2006). O modelo será reconstruído a partir do melhor trial persistido, mas as previsões usadas para avaliar os alertas serão produzidas por cinco modelos treinados em partes diferentes dos motores. Dessa forma, cada previsão será feita por um modelo que não viu aquele motor durante o treinamento.

In [2]:
train = add_rul_label(load_split(DATA_DIR, 'train', 'FD001'))
features = nonconstant_features(
    train,
    candidate_rul_features(include_settings=True),
)
study = optimize_model(
    train,
    features=features,
    model_name='Extra Trees',
    n_trials=100,
    n_splits=5,
    random_state=42,
    storage_dir=PROJECT_ROOT / 'outputs' / 'optuna',
    search_version='v1',
)
model = build_pipeline_from_params(
    model_name='Extra Trees',
    params=study.best_params,
    n_features=len(features),
    random_state=42,
    search_version='v1',
)
oof_predictions = generate_oof_rul_predictions(
    train,
    features=features,
    model=model,
    n_splits=5,
)
oof_predictions.head()

[I 2026-09-26 16:31:03,513] Using an existing study with `study_name='fd001_extra_trees_v1'` instead of creating a new one.


,unit,cycle,rul,rul_predito
0,1,1,191,201.990039
1,1,2,190,198.900466
2,1,3,189,199.315029
3,1,4,188,208.893386
4,1,5,187,195.227236


## 2. Política de alerta

Será considerado um horizonte operacional de 30 ciclos apenas como cenário de análise. Um alerta será considerado oportuno quando for confirmado dentro dos 30 ciclos anteriores à falha. Um alerta emitido antes dessa janela será classificado como prematuro, pois poderia provocar uma intervenção antes do momento planejado. Quando nenhum alerta for confirmado antes da falha, será registrada uma falha não detectada.

A confirmação exigirá três ciclos consecutivos abaixo do limiar. Essa regra simples reduz a influência de uma oscilação isolada do sensor ou da previsão. O horizonte, o limiar e a confirmação não são verdades universais; serão tratados como parâmetros que futuramente poderão receber custos diferentes.

In [3]:
warning_summary = evaluate_warning_grid(
    oof_predictions,
    thresholds=[10, 20, 30, 40, 50],
    warning_horizon=30,
    confirmation_cycles=3,
)
warning_summary.round(2)

,limiar_alerta,horizonte_alerta,ciclos_confirmacao,motores,alertas_oportunos,alertas_prematuros,falhas_nao_detectadas,taxa_alerta_oportuna,taxa_alerta_prematuro,taxa_falha_nao_detectada,lead_time_medio_oportuno,lead_time_mediano_oportuno
0,10,30,3,100.0,97.0,0.0,3.0,0.97,0.00,0.03,3.96,4.0
1,20,30,3,100.0,99.0,1.0,0.0,0.99,0.01,0.00,13.08,13.0
2,30,30,3,100.0,86.0,14.0,0.0,0.86,0.14,0.00,19.73,19.0
3,40,30,3,100.0,54.0,46.0,0.0,0.54,0.46,0.00,22.07,22.5
4,50,30,3,100.0,32.0,68.0,0.0,0.32,0.68,0.00,22.84,23.0


### Insights

A tabela permite observar o compromisso entre antecipação e confiabilidade. Um limiar mais alto tende a emitir avisos mais cedo, mas pode aumentar os alertas prematuros. Um limiar mais baixo tende a reduzir intervenções antecipadas, mas pode aumentar a quantidade de falhas não detectadas. Esse equilíbrio será a base da etapa seguinte, em que serão atribuídos custos diferentes a cada consequência.

In [4]:
rates = warning_summary.melt(
    id_vars=['limiar_alerta'],
    value_vars=['taxa_alerta_oportuna', 'taxa_alerta_prematuro', 'taxa_falha_nao_detectada'],
    var_name='resultado',
    value_name='taxa',
)
fig = px.line(
    rates,
    x='limiar_alerta',
    y='taxa',
    color='resultado',
    markers=True,
    labels={
        'limiar_alerta': 'Limiar de alerta (RUL em ciclos)',
        'taxa': 'Proporção de motores',
        'resultado': 'Resultado do alerta',
    },
    title=lang({'title': 'Trade-off entre alertas oportunos, prematuros e falhas não detectadas'})['title'],
    height=550,
)
fig.show()

### Insights

A figura não determina sozinha o melhor limiar. Ela apenas mostra as consequências operacionais de cada escolha. A decisão deverá ser tomada na próxima etapa, quando o custo de uma intervenção antecipada puder ser comparado ao custo de uma falha não detectada.

## Referências

GEURTS, P.; ERNST, D.; WEHENKEL, L. Extremely randomized trees. Machine Learning, v. 63, p. 3–42, 2006.

PEDREGOSA, F. et al. Scikit-learn: Machine Learning in Python. Journal of Machine Learning Research, v. 12, p. 2825–2830, 2011.